# 03 — Prepare & Export

Package the cleaned `films_adjusted` table as the published dataset: CSV
(+ Excel/Parquet) with a plain-English codebook for every column.

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from src.ingest import load_config
from src.clean_quality import get_connection
from src.prepare import package_dataset

cfg = load_config('config.yaml')
con = get_connection(cfg)
films = con.execute('SELECT * FROM films_adjusted ORDER BY adjusted_gross DESC').df()
worldwide = con.execute('SELECT * FROM films_worldwide ORDER BY worldwide_gross DESC').df()
genre = con.execute('SELECT * FROM films_genre ORDER BY title').df()
print('adjusted', films.shape, '| worldwide', worldwide.shape, '| genre', genre.shape)

## Codebook — a plain-English description for every column

In [ ]:
codebook = {
    'rank_adjusted':     'Rank by inflation-adjusted domestic gross (1 = highest).',
    'title':             'Film title.',
    'adjusted_gross':    'Domestic lifetime gross adjusted to 2022 dollars via ticket-price inflation (USD).',
    'nominal_gross':     'Domestic lifetime gross in year-of-release dollars, as originally reported (USD).',
    'est_tickets':       'Estimated number of tickets sold over the film lifetime (Box Office Mojo estimate).',
    'release_year':      'Year of the film original theatrical release.',
    'decade':            'Release decade (release_year rounded down to the nearest 10).',
    'inflation_multiple':'adjusted_gross / nominal_gross - how many times its original take the adjusted figure represents.',
}

notes = '''
Source: Box Office Mojo, Top Lifetime Adjusted Grosses (domestic, US/Canada), adjusted to 2022 dollars.
URL: https://www.boxofficemojo.com/chart/top_lifetime_gross_adjusted/?adjust_gross_to=2022

Method: adjusted gross = estimated tickets sold x the 2022 average ticket price (ticket-price inflation,
NOT CPI). Domestic only. A film lifetime total includes re-release grosses, which inflates some classics.
Nominal grosses are in year-of-release dollars and are not comparable across eras without the adjustment.
'''
written = package_dataset(films, cfg, name='highest_grossing_films_v1', codebook=codebook, notes=notes)
written

## Package the worldwide + genre exports

In [ ]:
ww_codebook = {
    'rank_worldwide': 'Rank by worldwide lifetime gross (1 = highest).',
    'title': 'Film title.',
    'worldwide_gross': 'Worldwide lifetime gross, nominal USD (domestic + foreign).',
    'domestic_gross': 'Domestic (U.S. & Canada) lifetime gross, nominal USD.',
    'foreign_gross': 'International (rest-of-world) lifetime gross, nominal USD.',
    'release_year': 'Year of original theatrical release.',
    'domestic_pct': 'Domestic gross as a percent of worldwide.',
    'foreign_pct': 'International gross as a percent of worldwide.',
}
ww_notes = '''Source: Box Office Mojo, Top Lifetime Grosses (Worldwide).
Nominal (year-of-release) dollars, NOT inflation-adjusted. Domestic = US & Canada.'''
package_dataset(worldwide, cfg, name='films_worldwide_v1', codebook=ww_codebook, notes=ww_notes)

genre_codebook = {
    'title': 'Film title.', 'release_year': 'Release year (match key).',
    'tmdb_id': 'TMDB movie id.', 'tmdb_title': 'Title as returned by TMDB.',
    'primary_genre': "TMDB's first-listed genre for the film.",
    'genres_str': 'All TMDB genres, comma-separated.',
    'matched': 'True if a TMDB result was found for title+year.',
}
genre_notes = '''Source: TMDB (themoviedb.org), matched by title + release year.
Genres are TMDB editorial tags; most films have several. Used as a ratio in the
genre market-split analysis (see SOURCES.md for the multi-genre attribution note).'''
package_dataset(genre, cfg, name='films_genre_v1', codebook=genre_codebook, notes=genre_notes)

---
**Next:** `04-viz.ipynb` (exploration) and `06-viz-social.ipynb` (the three social charts).

## Cleanup

In [ ]:
con.close()
print('connection closed')